# Ensemble Retrieval: Combining Semantic and Keyword Search with Custom LangChain Components

In advanced Retrieval-Augmented Generation (RAG) systems, relying on a single retrieval mechanism is often insufficient. A pure dense vector search excels at capturing semantic meaning—understanding that "automobile" relates to "car"—but can struggle when the query requires an exact keyword match or specific technical term. Conversely, traditional sparse methods like BM25 are excellent for precise keyword matching but fail entirely when the user's language is semantically related but lexically different from the source text. This inherent limitation of single-strategy retrievers necessitates a more robust approach: **ensemble retrieval**.

Ensemble retrieval solves this by combining the strengths of multiple specialized retrievers (e.g., vector search, BM25, graph traversal) into one cohesive system. The most common and effective method for fusing these results is using techniques like Reciprocal Rank Fusion (RRF). RRF mathematically combines the ranked lists from different sources, giving higher weight to documents that rank highly across *multiple* strategies, thereby significantly improving overall recall and precision without requiring complex re-ranking models.

By implementing a custom `BaseRetriever` class in LangChain, we move beyond simply calling off-the-shelf components. We learn how to architect the core logic of an advanced RAG component—a system that intelligently orchestrates multiple search pipelines and fuses their results using established ranking algorithms. Mastering this pattern is critical for building production-grade, highly accurate RAG applications that can handle diverse user queries in complex, multi-step LangGraph workflows.

### Learning Objectives
Upon completing this notebook, you will be able to:
*   **Understand Retrieval Limitations:** Articulate the specific failure modes of pure dense (semantic) and sparse (keyword) retrieval methods.
*   **Implement Ensemble Logic:** Design and implement a custom `BaseRetriever` component in LangChain.
*   **Apply Reciprocal Rank Fusion (RRF):** Understand and apply the mathematical principles of RRF to fuse ranked document lists from multiple sources.
*   **Build Hybrid Systems:** Successfully combine disparate retrieval strategies (e.g., ChromaDB vector search and BM25 keyword search) into a single, robust retriever for advanced RAG pipelines.


### Setup and Imports

This cell initializes the environment by loading necessary environment variables (`.env`) and imports core libraries for advanced retrieval tasks. It brings in components like `OpenAIEmbeddings` for vector representation, `Chroma` for vector storage, and specialized retrievers such as `BM25Retriever`.


In [2]:
import os
from typing import List
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks.manager import CallbackManagerForRetrieverRun
from langchain_community.retrievers import BM25Retriever

load_dotenv()


True

### Data Setup and Contextualization

This cell initializes a list of `Document` objects, simulating the corpus (the knowledge base) that the RAG system will query. It is crucial because it establishes a controlled environment where specific documents are designed to test different retrieval mechanisms: keyword matching (Docs 1-2), semantic relevance without keywords (Docs 3-5), and off-topic noise (Docs 6-12).


In [3]:
# Same 12 documents as the hybrid search notebook
# Docs 1-2: contain the exact word "vaccine" — BM25 keyword match
# Docs 3-5: semantically related (immune system, antibodies, herd immunity) but lack the word "vaccine"
#            — dense search finds these, BM25 misses them
# Docs 6-12: off-topic
docs = [
    Document(page_content="Vaccines work by introducing a weakened or inactivated pathogen to trigger an immune response.", metadata={"topic": "health"}),
    Document(page_content="The flu vaccine is reformulated each year to match the most prevalent circulating virus strains.", metadata={"topic": "health"}),
    Document(page_content="The immune system produces antibodies that recognise and neutralise foreign pathogens in the body.", metadata={"topic": "health"}),
    Document(page_content="Herd immunity occurs when enough of a population becomes resistant to a disease, slowing its spread.", metadata={"topic": "health"}),
    Document(page_content="White blood cells called B-lymphocytes produce proteins that bind to and destroy specific antigens.", metadata={"topic": "health"}),
    Document(page_content="Version control systems like Git track changes to code and enable collaboration across teams.", metadata={"topic": "programming"}),
    Document(page_content="Docker containers package applications with their dependencies for consistent deployment.", metadata={"topic": "programming"}),
    Document(page_content="The French Revolution began in 1789 and fundamentally transformed European political structures.", metadata={"topic": "history"}),
    Document(page_content="The Silk Road was an ancient trade network connecting China to the Mediterranean world.", metadata={"topic": "history"}),
    Document(page_content="The Amazon rainforest produces about 20% of the world's oxygen and houses 10% of all species.", metadata={"topic": "nature"}),
    Document(page_content="Coral reefs cover less than 1% of the ocean floor but support about 25% of all marine species.", metadata={"topic": "nature"}),
    Document(page_content="REST APIs communicate over HTTP using standard methods like GET, POST, PUT, and DELETE.", metadata={"topic": "programming"}),
]

### Ensemble Retriever Setup

This cell initializes two distinct retrieval mechanisms: a dense retriever (using ChromaDB and OpenAI embeddings) for semantic similarity, and a sparse retriever (using BM25Plus) for keyword matching. By setting up both, we create an ensemble approach that combines the strengths of embedding-based recall with traditional term frequency scoring.


In [4]:
# Dense retriever: ChromaDB with OpenAI embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Create a vector store in ChromaDB using the provided documents and embeddings.
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="custom_ensemble",
)

# Convert the vector store into a retriever object, configured to fetch the top 4 similar documents.
chroma_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

# Sparse retriever: BM25Plus variant on raw text, no embeddings
# BM25Plus ensures every matched term contributes a positive score,
# which improves recall for short documents like the ones we have here
# Initialize the BM25 retriever using the original documents.
bm25_retriever = BM25Retriever.from_documents(
    docs,
    k=2,
    bm25_variant="plus"
)


### Ensemble Retriever Implementation

This custom class, `MyEnsembleRetriever`, implements an advanced retrieval strategy by combining the results of multiple underlying retrievers. It uses Reciprocal Rank Fusion (RRF) to calculate a weighted score for each document across all sources, ensuring that the final ranking is a robust fusion of individual retriever performances.


In [5]:
class MyEnsembleRetriever(BaseRetriever):
    """Custom ensemble retriever that fuses results from multiple retrievers
    using Reciprocal Rank Fusion (RRF).

    RRF score for a document d across retriever i:
        score(d) = sum over i of [ weight_i * (1 / (rank_i(d) + rrf_k)) ]

    rrf_k is a smoothing constant (default 60) — it dampens the outsized
    advantage of rank-1 documents so lower-ranked results still contribute.
    Documents not returned by a retriever contribute 0 for that retriever.
    """

    retrievers: List[BaseRetriever]
    weights: List[float]
    rrf_k: int = 60

    def _get_relevant_documents(
        self, query: str, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Document]:
        # Collect ranked result lists from every retriever
        all_results: List[List[Document]] = [
            retriever.invoke(query) for retriever in self.retrievers
        ]

        # Accumulate RRF scores; key by page_content to deduplicate
        # doc_scores maps page_content -> (accumulated_score, Document)
        doc_scores: dict[str, tuple[float, Document]] = {}

        for retriever_idx, results in enumerate(all_results): # Iterate through each retriever's results
            weight = self.weights[retriever_idx] # Get the weight for the current retriever
            for rank, doc in enumerate(results):
                # Calculate RRF contribution: weight * (1 / (rank + rrf_k))
                rrf_score = weight * (1.0 / (rank + self.rrf_k)) 
                key = doc.page_content
                if key in doc_scores:   # Check if this document has already been scored
                    # Add contribution to existing score
                    prev_score, prev_doc = doc_scores[key]  # Fetch previous score and document object
                    doc_scores[key] = (prev_score + rrf_score, prev_doc)  # Update the score
                else:
                    # First time seeing this document, initialize its score
                    doc_scores[key] = (rrf_score, doc)

        # Sort by accumulated score descending and return the documents
        sorted_docs = sorted(doc_scores.values(), key=lambda x: x[0], reverse=True)
        return [doc for _, doc in sorted_docs]



### Ensemble Retriever Initialization

This cell instantiates a custom `MyEnsembleRetriever`. This specialized retriever combines the results of multiple underlying retrievers (here, Chroma and BM25) using weighted averaging. It is crucial for improving retrieval robustness by leveraging the strengths of different embedding/indexing methods.


In [6]:
# Instantiate our custom retriever 
my_ensemble = MyEnsembleRetriever(
    retrievers=[chroma_retriever, bm25_retriever], # Pass a list of individual retrievers to be combined
    weights=[0.8, 0.2], # Assign weights (e.g., Chroma gets 80%, BM25 gets 20%) to control their influence
    rrf_k=60, # Set the number of results to retrieve using Reciprocal Rank Fusion (RRF)
)



### Code Explanation: Ensemble Retrieval Demonstration

This cell demonstrates the core benefit of an ensemble retriever. It queries three different retrieval mechanisms (BM25, ChromaDB, and a custom fusion `my_ensemble`) using the same query to show how each method retrieves documents differently—keyword matching vs. semantic understanding—and finally, how the ensemble combines the strengths of both.


In [7]:
query = "How do vaccines work to protect against diseases?"

# 1. Invoke BM25 (Keyword-based retrieval)
bm25_results = bm25_retriever.invoke(query)
# 2. Invoke ChromaDB (Semantic/Vector-based retrieval)
chroma_results = chroma_retriever.invoke(query)
# 3. Invoke the custom ensemble retriever
ensemble_results = my_ensemble.invoke(query)

# BM25 matches on the exact word "vaccine" — finds docs 1 and 2
# but misses the semantically related immune/antibody docs (3, 4, 5)
print("=== BM25 Only (keyword match) ===")
for i, doc in enumerate(bm25_results, 1):
    print(f"  [{i}] topic={doc.metadata['topic']}: {doc.page_content}")

print()

# Dense search finds docs 3, 4, 5 through semantic understanding
# even though they don't contain the word "vaccine"
print("=== ChromaDB Only (semantic match) ===")
for i, doc in enumerate(chroma_results, 1):
    print(f"  [{i}] topic={doc.metadata['topic']}: {doc.page_content}")

print()

# Our custom RRF ensemble merges both ranked lists — keyword matches AND semantic matches surface together
print("=== MyEnsembleRetriever (custom RRF fusion) ===")
for i, doc in enumerate(ensemble_results, 1):
    print(f"  [{i}] topic={doc.metadata['topic']}: {doc.page_content}")


=== BM25 Only (keyword match) ===
  [1] topic=health: Vaccines work by introducing a weakened or inactivated pathogen to trigger an immune response.
  [2] topic=programming: REST APIs communicate over HTTP using standard methods like GET, POST, PUT, and DELETE.

=== ChromaDB Only (semantic match) ===
  [1] topic=health: Vaccines work by introducing a weakened or inactivated pathogen to trigger an immune response.
  [2] topic=health: The immune system produces antibodies that recognise and neutralise foreign pathogens in the body.
  [3] topic=health: The flu vaccine is reformulated each year to match the most prevalent circulating virus strains.
  [4] topic=health: White blood cells called B-lymphocytes produce proteins that bind to and destroy specific antigens.

=== MyEnsembleRetriever (custom RRF fusion) ===
  [1] topic=health: Vaccines work by introducing a weakened or inactivated pathogen to trigger an immune response.
  [2] topic=health: The immune system produces antibodies that 